# 02. Data Loading and Preparation

This notebook validates the MUSAE edge list, targets, and sparse content features, then constructs the undirected mutual-follow graph used by the remaining notebooks.


In [1]:
from pathlib import Path
import os
import sys
import warnings

import networkx as nx
import numpy as np
import pandas as pd
from IPython.display import Image, display


def find_project_root(start: Path) -> Path:
    """Find the repository root whether Jupyter starts there or in notebooks/."""
    for candidate in (start, *start.parents):
        if (candidate / "requirements.txt").exists() and (candidate / "data").exists():
            return candidate.resolve()
    raise FileNotFoundError("Could not locate the project root.")


ROOT = find_project_root(Path.cwd())
os.chdir(ROOT)
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

os.environ.setdefault("PYTHONHASHSEED", "0")
warnings.filterwarnings("ignore", category=FutureWarning)
SEED = 42
print(f"Project root: {ROOT}")


Project root: C:\Users\Jimi\Desktop\github-social-network


## Essential cleaning and graph-construction logic

Each edge is an undirected mutual-follow relation. The reusable loader in `src/data_loader.py` converts IDs to integers and raises an error for missing endpoints, self-loops, duplicate undirected edges, duplicate target IDs, invalid binary labels, or graph nodes without labels. The essential logic is:

```text
read edges and targets
require the expected columns and numeric IDs
reject missing values and self-loops
canonicalize every pair as (min(u,v), max(u,v)) and reject duplicates
require one binary label per node
G = undirected graph from the validated edge list
require every graph node to have a target
```


### Production implementation: validated data loading and graph construction

The following cell reproduces src/data_loader.py, which reads the edge and target CSV files, validates their schema and values, and builds the undirected NetworkX graph. Its inputs are file paths or an edge DataFrame; its outputs are the graph, validated tables, and the node-to-label mapping used throughout the study. The explicit error checks prevent malformed edges, labels, or missing graph labels from silently changing later statistics and model results.


In [2]:
from pathlib import Path
from typing import Dict, Tuple

import networkx as nx
import pandas as pd


EDGE_COLUMNS = ("id_1", "id_2")
TARGET_ID_COLUMN = "id"
TARGET_LABEL_COLUMN = "ml_target"


def load_edges(path: str | Path) -> pd.DataFrame:
    """Load and validate the edge list."""
    path = Path(path)
    edges_df = pd.read_csv(path)

    missing_columns = [col for col in EDGE_COLUMNS if col not in edges_df.columns]
    if missing_columns:
        raise ValueError(f"Edge file is missing required columns: {missing_columns}")

    edges_df = edges_df.loc[:, list(EDGE_COLUMNS)].copy()
    edges_df["id_1"] = pd.to_numeric(edges_df["id_1"], errors="coerce")
    edges_df["id_2"] = pd.to_numeric(edges_df["id_2"], errors="coerce")
    if edges_df[list(EDGE_COLUMNS)].isna().any().any():
        raise ValueError("Edge file contains missing or non-numeric endpoints.")
    edges_df = edges_df.astype({"id_1": "int64", "id_2": "int64"})

    if (edges_df["id_1"] == edges_df["id_2"]).any():
        raise ValueError("Edge file contains self-loops.")

    canonical_edges = pd.DataFrame(
        {
            "u": edges_df[["id_1", "id_2"]].min(axis=1),
            "v": edges_df[["id_1", "id_2"]].max(axis=1),
        }
    )
    if canonical_edges.duplicated().any():
        raise ValueError("Edge file contains duplicate undirected edges.")

    return edges_df


def load_targets(path: str | Path) -> pd.DataFrame:
    """Load and validate node labels, returning a standardized labels DataFrame."""
    path = Path(path)
    targets_df = pd.read_csv(path)

    required_columns = [TARGET_ID_COLUMN, TARGET_LABEL_COLUMN]
    missing_columns = [col for col in required_columns if col not in targets_df.columns]
    if missing_columns:
        raise ValueError(f"Target file is missing required columns: {missing_columns}")

    keep_columns = [TARGET_ID_COLUMN, TARGET_LABEL_COLUMN]
    if "name" in targets_df.columns:
        keep_columns.insert(1, "name")

    labels_df = targets_df.loc[:, keep_columns].copy()
    labels_df[TARGET_ID_COLUMN] = pd.to_numeric(labels_df[TARGET_ID_COLUMN], errors="coerce")
    labels_df[TARGET_LABEL_COLUMN] = pd.to_numeric(labels_df[TARGET_LABEL_COLUMN], errors="coerce")
    if labels_df[[TARGET_ID_COLUMN, TARGET_LABEL_COLUMN]].isna().any().any():
        raise ValueError("Target file contains missing or non-numeric IDs or labels.")
    labels_df[TARGET_ID_COLUMN] = labels_df[TARGET_ID_COLUMN].astype("int64")
    labels_df[TARGET_LABEL_COLUMN] = labels_df[TARGET_LABEL_COLUMN].astype("int64")
    if labels_df[TARGET_ID_COLUMN].duplicated().any():
        raise ValueError("Target file contains duplicate node IDs.")

    unique_labels = set(labels_df[TARGET_LABEL_COLUMN].unique())
    if not unique_labels.issubset({0, 1}):
        raise ValueError(f"Unexpected labels detected in target file: {sorted(unique_labels)}")

    labels_df = labels_df.rename(columns={TARGET_LABEL_COLUMN: "label"})

    return labels_df


def build_graph(edges_df: pd.DataFrame) -> nx.Graph:
    """Build an undirected NetworkX graph from an edges DataFrame."""
    missing_columns = [col for col in EDGE_COLUMNS if col not in edges_df.columns]
    if missing_columns:
        raise ValueError(f"Edges DataFrame is missing required columns: {missing_columns}")

    graph = nx.from_pandas_edgelist(edges_df, source="id_1", target="id_2", create_using=nx.Graph())
    return graph


def load_dataset(data_dir: str | Path) -> Tuple[nx.Graph, pd.DataFrame, pd.DataFrame, Dict[int, int]]:
    """
    Load the dataset from a directory and build all base objects.

    Returns:
        G: Undirected graph
        edges_df: Edge list DataFrame
        labels_df: Labels DataFrame with standardized 'label' column
        labels_dict: Mapping node_id -> label
    """
    data_dir = Path(data_dir)
    edges_path = data_dir / "musae_git_edges.csv"
    targets_path = data_dir / "musae_git_target.csv"

    if not edges_path.exists():
        raise FileNotFoundError(f"Missing edge file: {edges_path}")
    if not targets_path.exists():
        raise FileNotFoundError(f"Missing target file: {targets_path}")

    edges_df = load_edges(edges_path)
    labels_df = load_targets(targets_path)
    graph = build_graph(edges_df)

    labels_dict = dict(zip(labels_df["id"], labels_df["label"]))
    missing_labels = set(graph.nodes()) - set(labels_dict)
    if missing_labels:
        raise ValueError(f"Graph contains {len(missing_labels)} nodes without labels.")
    nx.set_node_attributes(graph, labels_dict, name="label")

    return graph, edges_df, labels_df, labels_dict


In [3]:
from content_features import load_content_features
from data_loader import load_dataset

G, edges_df, labels_df, labels_dict = load_dataset(ROOT / "data")
content_dict, n_content_features = load_content_features(
    ROOT / "data" / "musae_git_features.json"
)

canonical_edges = pd.DataFrame({
    "u": edges_df[["id_1", "id_2"]].min(axis=1),
    "v": edges_df[["id_1", "id_2"]].max(axis=1),
})
validation = pd.DataFrame([{
    "nodes": G.number_of_nodes(),
    "edges": G.number_of_edges(),
    "connected_components": nx.number_connected_components(G),
    "self_loops": nx.number_of_selfloops(G),
    "duplicate_undirected_edges": int(canonical_edges.duplicated().sum()),
    "labeled_nodes": len(labels_dict),
    "content_nodes": len(content_dict),
    "content_feature_dimensions": n_content_features,
}])
display(validation)
display(edges_df.head())
display(labels_df.head())

assert validation.iloc[0].to_dict() == {
    "nodes": 37700,
    "edges": 289003,
    "connected_components": 1,
    "self_loops": 0,
    "duplicate_undirected_edges": 0,
    "labeled_nodes": 37700,
    "content_nodes": 37700,
    "content_feature_dimensions": 4005,
}
assert set(G) == set(labels_dict) == set(content_dict)


,nodes,edges,connected_components,self_loops,duplicate_undirected_edges,labeled_nodes,content_nodes,content_feature_dimensions
0,37700,289003,1,0,0,37700,37700,4005


,id_1,id_2
0,0,23977
1,1,34526
2,1,2370
3,1,14683
4,1,29982


,id,name,label
0,0,Eiryyy,0
1,1,shawflying,0
2,2,JpMCarrilho,1
3,3,SuhwanCha,0
4,4,sunilangadi2,1


The validated graph has 37,700 labeled nodes and 289,003 edges in one connected component. All 37,700 nodes also occur in the content-feature mapping. The cleaning rules reject invalid input instead of silently dropping or repairing rows, so a malformed dataset cannot change the graph unnoticed.

## Summary

This Notebook loaded and strictly validated the MUSAE edge list, target labels, and sparse content-feature mapping. The checks rejected missing or nonnumeric identifiers, self-loops, duplicate undirected edges, duplicate targets, invalid binary labels, absent files, and graph nodes without labels before constructing the undirected NetworkX graph.

The validated dataset contained 37,700 nodes and 289,003 edges in one connected component, with no self-loops or duplicate undirected edges. All 37,700 graph nodes had both a binary field label and a content-feature record, and the content representation contained 4,005 dimensions.

The resulting graph, edge table, standardized label table, node-to-label mapping, and content-feature mapping were aligned exactly. These validated objects form the common input for graph statistics, communities, classification, embeddings, link reconstruction, and null-model analysis.
